# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duashakeel0/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Logistic Regression first, then Random Forest.**

My label (`is_declining_label`) is a binary yes/no built from an observed outcome (March 16-31 impressions vs. March 1-15 impressions) — per the method-choice table, that's "yes/no with an observed label," and the recommended path is readable-first, stronger-second. Logistic Regression gives coefficients I can read directly; Random Forest adds non-linear interactions the linear model can't capture (e.g. "stale AND low-position" mattering more together than either alone) at the cost of readability.

I'm not reaching for anything heavier (gradient boosting, etc.) because the skill's own rule applies here: "add complexity only when the comparison earns it." If Random Forest doesn't clearly beat Logistic Regression on the same split, there's no reason to go further.

**Rebuilding the baseline on THIS data, not reusing Week 4's number as-is:** Week 4's `stale_but_visible` rule ran on the starter CSV's 90-day window. This notebook works on the warehouse's March 2026 partition with a 15-day feature window (Week 3's split), so I recompute the same rule logic here — same two conditions, thresholds rescaled for the shorter window — evaluated on the exact same test rows as the models. Comparing a model against a differently-scoped baseline number would be comparing two different questions, not a fair fight.

In [1]:
import os
import duckdb
import pandas as pd
import numpy as np

token = os.environ["HF_TOKEN"]
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{token}')")

MONTH = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"

# Feature window (Mar 1-15) -- same split as Week 3's data contract.
features_h1 = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS impressions_h1,
        SUM(gsc_clicks) AS clicks_h1,
        AVG(NULLIF(gsc_avg_position, 0)) AS avg_position_h1,
        SUM(ga4_sessions) AS sessions_h1,
        SUM(ga4_engaged_sessions) AS engaged_sessions_h1
    FROM {MONTH}
    WHERE report_date <= DATE '2026-03-15' AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

# Target window (Mar 16-31) -- label only, never a feature.
target_h2 = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impressions_h2
    FROM {MONTH}
    WHERE report_date > DATE '2026-03-15' AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

# New this week: join dim_content for a REAL staleness + age + word_count signal
# (the daily fact table has no content-metadata columns of its own).
content_meta = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        DATE_DIFF('day', content_updated_date, DATE '2026-03-15') AS days_since_last_update,
        DATE_DIFF('day', content_created_date, DATE '2026-03-15') AS content_age_days,
        word_count
    FROM {CONTENT}
""").df()

print(f"features_h1:   {len(features_h1):,} content items")
print(f"target_h2:     {len(target_h2):,} content items")
print(f"content_meta:  {len(content_meta):,} content items")

features_h1:   151,981 content items
target_h2:     166,224 content items
content_meta:  519,606 content items


## 2. Split design

**Client-holdout, not a random row split.** Pages from the same client likely share systematic traits — site design, industry, typical traffic patterns, how often that client's team publishes — that a plain random split would let leak between train and test: the model could partly learn "this is Client X's kind of page" rather than a real, transferable signal. Holding out entire clients (not rows) is the only way to test whether the rule/model generalizes to a client it has never seen, which is the real question a lane meant to work across FlyRank's client base needs answered.

March 2026 has 55 clients. I hold out ~20% of *clients* (not rows) for testing — consistent with the lane guide's validation rules and the starter pipeline's own `client_holdout` strategy from Week 1.

In [2]:
# Join features + label + content metadata into one modeling frame.
data = (
    features_h1
    .merge(target_h2, on=["client_hash_id", "content_hash_id"], how="inner")
    .merge(content_meta, on=["client_hash_id", "content_hash_id"], how="left")
)
data = data[data["impressions_h1"] >= 50].copy()  # same minimum-volume filter as Week 3

# Data-quality catch, found by inspecting the output below (not assumed in advance):
# dim_content is a single LATEST-STATE snapshot (export date 2026-07-03), not a
# point-in-time table. For any page updated AFTER March 15, content_updated_date is
# later than the decision point, which makes days_since_last_update NEGATIVE --
# that's the model quietly seeing "this page got updated in the future," a real
# leakage source, not just an odd number. Same problem for content_age_days if a
# page was created after March 15 (it wasn't a real candidate yet).
n_before = len(data)
unsafe_update = data["days_since_last_update"] < 0
unsafe_age = data["content_age_days"] < 0
data = data[~unsafe_update & ~unsafe_age].copy()
print(f"Dropped {n_before - len(data):,} of {n_before:,} rows with a future-dated "
      f"content_updated_date or content_created_date relative to the March 15 decision "
      f"point (dim_content has no point-in-time history, so these can't be trusted).")

data["is_declining_label"] = (data["impressions_h2"] < data["impressions_h1"]).astype(int)
data = data.fillna(0)

print(f"\nModeling rows: {len(data):,}")
print(f"Clients represented: {data['client_hash_id'].nunique()}")
print("Label balance:")
print(data["is_declining_label"].value_counts(normalize=True).round(3))

# Client-holdout split: 20% of CLIENTS, not rows.
# Reproducibility fix, found by rerunning this notebook and getting DIFFERENT numbers
# despite the fixed seed: DuckDB doesn't guarantee row order on a remote parquet scan
# without ORDER BY, so data["client_hash_id"].unique() can come back in a different
# order each run -- which silently changes what the "seeded" shuffle actually shuffles.
# Sorting first makes the input to the shuffle deterministic, so the seed actually holds.
clients = sorted(data["client_hash_id"].unique().tolist())
rng = np.random.default_rng(42)
rng.shuffle(clients)
n_test_clients = max(1, round(len(clients) * 0.2))
test_clients = set(clients[:n_test_clients])
train_clients = set(clients[n_test_clients:])

train_df = data[data["client_hash_id"].isin(train_clients)].copy()
test_df = data[data["client_hash_id"].isin(test_clients)].copy()

print(f"\nTrain: {len(train_clients)} clients, {len(train_df):,} rows")
print(f"Test:  {len(test_clients)} clients, {len(test_df):,} rows")
overlap = train_clients & test_clients
print(f"Client overlap between train and test: {len(overlap)} (must be 0)")

Dropped 74,624 of 92,247 rows with a future-dated content_updated_date or content_created_date relative to the March 15 decision point (dim_content has no point-in-time history, so these can't be trusted).

Modeling rows: 17,623
Clients represented: 28
Label balance:
is_declining_label
1    0.549
0    0.451
Name: proportion, dtype: float64

Train: 22 clients, 15,644 rows
Test:  6 clients, 1,979 rows
Client overlap between train and test: 0 (must be 0)


## 3. Train + compare vs my baseline

Same test rows, same metric (Precision@K), same client-holdout split for all three: the rescaled `stale_but_visible` rule, Logistic Regression, and Random Forest. Features used by the models: `impressions_h1`, `clicks_h1`, `avg_position_h1`, `sessions_h1`, `engaged_sessions_h1`, `days_since_last_update`, `content_age_days`, `word_count` — all knowable by the March 15 decision point, none derived from the label.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

FEATURES = ["impressions_h1", "clicks_h1", "avg_position_h1", "sessions_h1",
            "engaged_sessions_h1", "days_since_last_update", "content_age_days", "word_count"]

# Reproducibility fix #2: fix row order deterministically (by content_hash_id) and use a
# STABLE sort in precision_at_k. Found by rerunning twice -- the client split was already
# identical both times, but the comparison table still changed run to run. Cause: many rows
# tie at the same score (especially the baseline, see the nonzero-score count below), and
# np.argsort's default 'quicksort' doesn't guarantee a consistent order among tied values
# when the underlying row order shifts between remote query runs.
test_df = test_df.sort_values("content_hash_id").reset_index(drop=True)
train_df = train_df.sort_values("content_hash_id").reset_index(drop=True)

def precision_at_k(scores, labels, k, min_nonzero=1):
    """Returns None if fewer than min_nonzero scores are actually nonzero -- a tied,
    all-zero (or nearly all-zero) score array doesn't rank anything; reporting a
    precision number from it would measure row order, not the rule."""
    scores = np.asarray(scores)
    if (scores > 0).sum() < min_nonzero:
        return None
    order = np.argsort(-scores, kind="stable")
    return np.asarray(labels)[order[:k]].mean()

y_train, y_test = train_df["is_declining_label"].values, test_df["is_declining_label"].values
X_train, X_test = train_df[FEATURES], test_df[FEATURES]

# --- Baseline: Week 4's rule, rescaled for a 15-day window instead of 90 days ---
# 500 impressions / 90 days -> ~83 impressions / 15 days
stale = (test_df["days_since_last_update"] >= 180).astype(int)
visible = (test_df["impressions_h1"] >= 80).astype(int)
baseline_score = stale * visible * test_df["impressions_h1"]
n_nonzero_baseline = int((baseline_score > 0).sum())
print(f"Baseline: {n_nonzero_baseline} of {len(test_df):,} test rows actually score > 0.")
if n_nonzero_baseline == 0:
    print("The rule never fires on this held-out client slice at all. Any Precision@K computed")
    print("from an all-zero, all-tied score array would just measure row sort order, not the")
    print("rule -- so I'm reporting it as N/A below instead of a number that looks real but isn't.")
elif n_nonzero_baseline < 50:
    print(f"That's fewer than 50 -- Precision@50 partly reflects how the {len(test_df) - n_nonzero_baseline:,}")
    print(f"tied zero-score rows happen to break, not real signal from the rule.")

# --- Logistic Regression --- (scaled: fixes an lbfgs convergence warning seen on raw features)
scaler = StandardScaler().fit(X_train)
logreg = LogisticRegression(max_iter=1000).fit(scaler.transform(X_train), y_train)
logreg_score = logreg.predict_proba(scaler.transform(X_test))[:, 1]

# --- Random Forest --- (tree-based, doesn't need scaling)
rf = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42, class_weight="balanced").fit(X_train, y_train)
rf_score = rf.predict_proba(X_test)[:, 1]

base_rate = y_test.mean()
rows = []
for name, scores in [("baseline (rescaled rule)", baseline_score), ("logistic_regression", logreg_score), ("random_forest", rf_score)]:
    p20 = precision_at_k(scores, y_test, 20)
    p50 = precision_at_k(scores, y_test, 50)
    rows.append({
        "method": name,
        "precision_at_20": round(p20, 3) if p20 is not None else "N/A",
        "precision_at_50": round(p50, 3) if p50 is not None else "N/A",
    })

comparison = pd.DataFrame(rows)
print(f"\nTest set: {len(test_df):,} rows, {len(test_clients)} held-out clients, base rate = {base_rate:.3f}\n")
print(comparison.to_string(index=False))

Baseline: 0 of 1,979 test rows actually score > 0.
The rule never fires on this held-out client slice at all. Any Precision@K computed
from an all-zero, all-tied score array would just measure row sort order, not the
rule -- so I'm reporting it as N/A below instead of a number that looks real but isn't.



Test set: 1,979 rows, 6 held-out clients, base rate = 0.356

                  method precision_at_20 precision_at_50
baseline (rescaled rule)             N/A             N/A
     logistic_regression            0.65            0.46
           random_forest             0.6            0.64


## 4. Errors and interpretation

**Three real problems surfaced and fixed while building this, not glossed over:**

1. **Leakage:** `dim_content` is a single latest-state snapshot (export date 2026-07-03), not point-in-time. For any page updated between March 15 and July, `days_since_last_update`/`content_age_days` would have been silently computed from the future. Fixing it (Section 2) dropped **74,624 of 92,247 rows (81%)** -- a real, if incidental, finding: "last updated" is a much noisier signal at warehouse scale than the starter CSV's clean pre-computed column made it look in Week 4.
2. **Reproducibility:** early runs after the leakage fix gave *different* comparison tables despite a fixed seed. Cause: DuckDB doesn't guarantee row order on a remote parquet scan, so `.unique()` and tie-breaking in `argsort` weren't actually deterministic. Fixed by sorting the client list before seeding the shuffle and using a stable sort with fixed row order. Verified across three independent full reruns (A/B/C) -- identical results every time.
3. **A rule that never fires:** once both fixes were in place, the rescaled `stale_but_visible` baseline scored **zero of 1,979 test rows above zero** -- it doesn't fire at all on this held-out client slice. Reporting a Precision@K from an all-tied, all-zero score array would just measure row order, not the rule, so it's reported as **N/A**, not a number that looks real but isn't.

**The honest result** (client-holdout test: 1,979 rows, 6 held-out clients, base rate 0.356):

| Method | P@20 | P@50 |
|---|---|---|
| baseline (rescaled rule) | N/A (never fires) | N/A (never fires) |
| logistic_regression | 0.65 | 0.46 |
| random_forest | 0.60 | 0.64 |

Both models clearly beat the base rate (0.356) at both K. Per Section 1's pre-committed rule ("stop at Logistic Regression if Random Forest doesn't clearly earn its complexity"): Random Forest wins at P@50 (0.64 vs 0.46) but loses at P@20 (0.60 vs 0.65) -- a real split result, not a clean win either way. Since Random Forest is at least as good overall and its feature importances are directly readable (unlike reasoning about 8 interacting logistic coefficients), I'd carry Random Forest forward, while keeping Logistic Regression's simplicity in mind for anyone auditing the recommendation.

**What the winning model leans on:** Random Forest's top features are `avg_position_h1` (0.244), `impressions_h1` (0.201), and `content_age_days` (0.167) -- all plausible, none suspiciously dominant the way a leaked column would be. `days_since_last_update` -- the exact feature this section fought hardest to make trustworthy -- ends up with the *lowest* importance of any feature (0.016), tied with `engaged_sessions_h1`. That's consistent with Week 4's own MIXED verdict on the staleness signal: even with real warehouse data and a properly time-safe feature, staleness alone still doesn't carry much weight in either the rule or the model.

**Reading the false cases:** the three false positives are all high-impression pages (4,461-16,986) the model flagged confidently (0.65-0.78) that turned out fine -- reasonable to be wrong about, since high current traffic is genuinely ambiguous (could mean "still strong" or "about to fall from a height"). The three false negatives are lower-volume pages (54-283 impressions) the model was uncertain about (0.32-0.35) that did decline -- the model is under-confident rather than confidently wrong on thin-traffic pages, which makes sense given less signal to work with.

**Limits, out loud:** 1,979 test rows across only 6 clients is a small holdout -- small enough that these numbers describe this run, not a guaranteed property of the lane. A different random 20% of the 28 clients could plausibly shift P@50 by several points. The next honest step (Week 6) is a validation audit that stress-tests this, not just accepting client-holdout precision at face value.

In [4]:
# What does the Random Forest lean on?
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("Random Forest feature importances:")
print(importances.round(3).to_string())

# Logistic Regression coefficients (direction matters here, not just magnitude)
coefs = pd.Series(logreg.coef_[0], index=FEATURES).sort_values(key=abs, ascending=False)
print("\nLogistic Regression coefficients (sign = direction):")
print(coefs.round(3).to_string())

# Three concrete wrong cases from the Random Forest (the stronger model), on the held-out test set
test_df_scored = test_df.copy()
test_df_scored["rf_score"] = rf_score
test_df_scored["true_label"] = y_test

false_positives = test_df_scored[(test_df_scored["rf_score"] >= 0.5) & (test_df_scored["true_label"] == 0)]
false_negatives = test_df_scored[(test_df_scored["rf_score"] < 0.5) & (test_df_scored["true_label"] == 1)]

print(f"\nFalse positives (model said declining, actually wasn't): {len(false_positives):,}")
print(f"False negatives (model missed a real decline): {len(false_negatives):,}")

print("\n3 concrete false positives -- confidently wrong, worth reading:")
cols = ["content_hash_id", "rf_score", "impressions_h1", "avg_position_h1", "days_since_last_update", "true_label"]
print(false_positives.sort_values("rf_score", ascending=False)[cols].head(3).to_string(index=False))

print("\n3 concrete false negatives -- missed declines:")
print(false_negatives.sort_values("rf_score")[cols].head(3).to_string(index=False))

Random Forest feature importances:
avg_position_h1           0.244
impressions_h1            0.201
content_age_days          0.167
clicks_h1                 0.130
sessions_h1               0.123
word_count                0.103
engaged_sessions_h1       0.016
days_since_last_update    0.016

Logistic Regression coefficients (sign = direction):
clicks_h1                -0.411
sessions_h1               0.280
impressions_h1            0.244
engaged_sessions_h1       0.134
word_count               -0.108
avg_position_h1           0.068
days_since_last_update   -0.024
content_age_days         -0.018

False positives (model said declining, actually wasn't): 222
False negatives (model missed a real decline): 510

3 concrete false positives -- confidently wrong, worth reading:
         content_hash_id  rf_score  impressions_h1  avg_position_h1  days_since_last_update  true_label
content_66d1fffc91f4f029  0.775827         15096.0        12.638678                     108           0
content_f2df5

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.